In [58]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets
from torchvision.transforms import ToTensor
import pandas as pd
import numpy as np
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, cross_validate, cross_val_score, cross_val_predict, \
                        GridSearchCV, RandomizedSearchCV, KFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder, LabelEncoder,PolynomialFeatures, TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.multiclass import OneVsRestClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, \
                            roc_curve, auc, roc_auc_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt 
import seaborn as sns 
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision import datasets, models, transforms
import os

In [3]:
train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

In [27]:
train["Churn"]

0          No
1          No
2          No
3         Yes
4         Yes
         ... 
594189     No
594190     No
594191     No
594192     No
594193    Yes
Name: Churn, Length: 594194, dtype: object

In [30]:
train.info()
mycols = ["gender","Partner","Dependents","PhoneService","MultipleLines","InternetService","OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies","Contract","PaperlessBilling","PaymentMethod"]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 594194 entries, 0 to 594193
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                594194 non-null  int64  
 1   gender            594194 non-null  object 
 2   SeniorCitizen     594194 non-null  int64  
 3   Partner           594194 non-null  object 
 4   Dependents        594194 non-null  object 
 5   tenure            594194 non-null  int64  
 6   PhoneService      594194 non-null  object 
 7   MultipleLines     594194 non-null  object 
 8   InternetService   594194 non-null  object 
 9   OnlineSecurity    594194 non-null  object 
 10  OnlineBackup      594194 non-null  object 
 11  DeviceProtection  594194 non-null  object 
 12  TechSupport       594194 non-null  object 
 13  StreamingTV       594194 non-null  object 
 14  StreamingMovies   594194 non-null  object 
 15  Contract          594194 non-null  object 
 16  PaperlessBilling  59

In [32]:
type(mycols)

list

In [37]:
train_X = train.drop('Churn',axis=1)
train_y = train['Churn']

In [43]:
t_enc = TargetEncoder()
train = t_enc.fit_transform(train_X,train_y)
test = t_enc.transform(test)

C:\Users\eSports\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but TargetEncoder was fitted without feature names
  warnings.warn(


In [54]:
train_X = torch.tensor(train)
test = torch.tensor(test)

C:\Users\eSports\AppData\Local\Temp\ipykernel_26128\3873672195.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_X = torch.tensor(train)
C:\Users\eSports\AppData\Local\Temp\ipykernel_26128\3873672195.py:2: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test = torch.tensor(test)


In [59]:
train = TensorDataset(train_X,train_y)
train,val =  random_split(train,[0.8,0.2])

In [60]:
train_loader = DataLoader(train,batch_size=32,shuffle=True)
val_loader = DataLoader(val,batch_size=32,shuffle=False)

In [61]:
model = nn.Sequential(
    nn.Linear(20,100),
    nn.ReLU(),
    nn.Linear(100,100),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(100,2)
)


criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)


In [ ]:
history = {"Train Acc":[],"Train Loss": [], "Val Acc":[], "Val Loss":[]}
epochs = 100
for epoch in epochs:
    running_loss = 0
    correct = 0
    total = 0
    model.train()
    for X_batch,y_batch in train_loader:
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs,y_batch)
        